# Отладка модуля защиты и итоговая оценка производительности:

In [ ]:
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Optional
import json
import re
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.transforms import functional as TF
from IPython.display import display
plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_columns", 200)

In [ ]:
@dataclass
class DefenseConfig:
    ROOT_DIR: str = "."

    # Пути к модели
    WEIGHTS_PATH: str = "tl_custom_train/resnet18_tl_custom_state_dict.pt"
    META_PATH: str = "tl_custom_train/resnet18_tl_custom_meta.json"
    ATTACK_RESULTS_DIR: str = "attack_experiments"
    OUT_DIR: str = "defense_experiments"

    # Какие атаки и какие eps анализировать
    ATTACK_METHODS: tuple = ("bim", "pgd")
    EPSILONS: tuple = (2/255, 4/255, 8/255)

    # Параметры генерации атак
    BIM_ALPHA: float = 1/255
    BIM_STEPS: int = 10

    PGD_ALPHA: float = 1/255
    PGD_STEPS: int = 10
    PGD_RANDOM_START: bool = True

    # Параметры оценивания
    BATCH_SIZE: int = 32
    NUM_WORKERS: int = 0
    MAX_SAMPLES_PER_ATTACK: Optional[int] = None
    SEED: int = 42

    # Защитные фильтры
    BLUR_KERNELS: tuple = (3, 5)
    QUANT_LEVELS: tuple = (64, 32)
    INCLUDE_COMBINED_DEFENSES: bool = True

    # Визуализации
    VIS_SAMPLES_PER_SETTING: int = 5

    # Опционально: визуализация защищённого ROI в исходном кадре
    CASCADE_ROOT: str = "runs_cascade_yolov8s"
    VIDEOS_DIR: str = "Videos"
    PAD_RATIO: float = 0.22

cfg = DefenseConfig()
ROOT = Path(cfg.ROOT_DIR)
OUT_DIR = ROOT / cfg.OUT_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)
random.seed(cfg.SEED)
np.random.seed(cfg.SEED)
torch.manual_seed(cfg.SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.SEED)
print(asdict(cfg))

## 1. Загрузка метаданных и модели

In [ ]:
DEFAULT_CLASS_TO_IDX = {"red": 0, "green": 1}
DEFAULT_INPUT_SIZE = 96
DEFAULT_MEAN = [0.485, 0.456, 0.406]
DEFAULT_STD = [0.229, 0.224, 0.225]

meta_path = ROOT / cfg.META_PATH
weights_path = ROOT / cfg.WEIGHTS_PATH

if meta_path.exists():
    with open(meta_path, "r", encoding="utf-8") as f:
        meta = json.load(f)
else:
    meta = {}

CLASS_TO_IDX = meta.get("class_to_idx", DEFAULT_CLASS_TO_IDX)
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}
NUM_CLASSES = len(CLASS_TO_IDX)

INPUT_SIZE = int(meta.get("input_size", DEFAULT_INPUT_SIZE))
MEAN = meta.get("imagenet_mean", meta.get("mean", DEFAULT_MEAN))
STD = meta.get("imagenet_std", meta.get("std", DEFAULT_STD))

print("CLASS_TO_IDX:", CLASS_TO_IDX)
print("INPUT_SIZE:", INPUT_SIZE)
print("MEAN:", MEAN)
print("STD:", STD)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.CrossEntropyLoss()

model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

state = torch.load(weights_path, map_location="cpu")
if isinstance(state, dict) and "state_dict" in state:
    state = state["state_dict"]

clean_state = {}
for k, v in state.items():
    nk = k.replace("module.", "")
    clean_state[nk] = v

missing, unexpected = model.load_state_dict(clean_state, strict=False)
model.to(device)
model.eval()

print("device:", device)
print("missing keys:", missing)
print("unexpected keys:", unexpected)

## 2. Поиск результатов атак

In [ ]:
ATTACK_DIR = ROOT / cfg.ATTACK_RESULTS_DIR
ATTACK_PATTERN = re.compile(r"^(fgsm|bim|pgd)_eps_(\d+)$", re.IGNORECASE)

def discover_attack_csvs(attack_dir: Path):
    found = {}
    if not attack_dir.exists():
        raise FileNotFoundError(f"Папка с результатами атак не найдена: {attack_dir}")

    for csv_path in attack_dir.glob("*.csv"):
        stem = csv_path.stem.lower()
        m = ATTACK_PATTERN.match(stem)
        if not m:
            continue
        method = m.group(1).lower()
        eps_px = int(m.group(2))
        found[(method, eps_px)] = csv_path
    return found

attack_csv_map = discover_attack_csvs(ATTACK_DIR)
print("Найдено attack CSV:", len(attack_csv_map))
for k, v in sorted(attack_csv_map.items()):
    print(k, "->", v.name)

In [ ]:
def load_attack_table(csv_path: Path):
    df = pd.read_csv(csv_path)
    if "crop_path" not in df.columns:
        raise ValueError(f"В {csv_path.name} нет колонки crop_path")
    if "target" not in df.columns:
        if "label" in df.columns:
            df["target"] = df["label"].map(CLASS_TO_IDX)
        else:
            raise ValueError(f"В {csv_path.name} нет ни target, ни label")
    if "label" not in df.columns:
        df["label"] = df["target"].map(IDX_TO_CLASS)
    df["crop_path"] = df["crop_path"].astype(str)
    return df

selected_tables = {}
for method in cfg.ATTACK_METHODS:
    for eps in cfg.EPSILONS:
        eps_px = int(round(eps * 255))
        key = (method.lower(), eps_px)
        if key in attack_csv_map:
            selected_tables[key] = load_attack_table(attack_csv_map[key])
        else:
            print(f"⚠ Не найден CSV для {key}")

if not selected_tables:
    raise RuntimeError("Не найдено ни одного attack CSV под текущие cfg.ATTACK_METHODS и cfg.EPSILONS")

summary_rows = []
for (method, eps_px), df in selected_tables.items():
    summary_rows.append({
        "method": method.upper(),
        "epsilon": f"{eps_px}/255",
        "n_rows": len(df),
        "csv": attack_csv_map[(method, eps_px)].name,
    })

attack_inventory_df = pd.DataFrame(summary_rows).sort_values(["method", "epsilon"]).reset_index(drop=True)
display(attack_inventory_df)

## 3. Dataset и функции генерации атак / защит

In [ ]:
val_tf = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

class AttackROIDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["crop_path"]).convert("RGB")
        x = self.transform(img)
        y = int(row["target"])
        return x, y, row["crop_path"], str(row["label"])

def make_loader(df, shuffle=False):
    ds = AttackROIDataset(df, val_tf)
    return DataLoader(
        ds,
        batch_size=cfg.BATCH_SIZE,
        shuffle=shuffle,
        num_workers=cfg.NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

mean_t = torch.tensor(MEAN, dtype=torch.float32, device=device).view(1, 3, 1, 1)
std_t = torch.tensor(STD, dtype=torch.float32, device=device).view(1, 3, 1, 1)
lower_bound = (0.0 - mean_t) / std_t
upper_bound = (1.0 - mean_t) / std_t

def sync_device():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def eps_to_tensor(eps):
    return torch.tensor([eps / s for s in STD], dtype=torch.float32, device=device).view(1, 3, 1, 1)

def denormalize_batch(x):
    return torch.clamp(x * std_t.to(x.device) + mean_t.to(x.device), 0.0, 1.0)

def normalize_batch(x01):
    mean = mean_t.to(x01.device)
    std = std_t.to(x01.device)
    return (x01 - mean) / std

def clamp_normalized(x):
    return torch.max(torch.min(x, upper_bound.to(x.device)), lower_bound.to(x.device))

def fgsm_attack(model, x, y, eps):
    x_orig = x.detach()
    eps_t = eps_to_tensor(eps)

    x_adv = x_orig.clone().detach().requires_grad_(True)
    logits = model(x_adv)
    loss = criterion(logits, y)

    model.zero_grad(set_to_none=True)
    loss.backward()

    grad_sign = x_adv.grad.sign()
    x_adv = x_adv + eps_t * grad_sign
    x_adv = torch.max(torch.min(x_adv, x_orig + eps_t), x_orig - eps_t)
    x_adv = clamp_normalized(x_adv).detach()
    return x_adv

def pgd_attack(model, x, y, eps, alpha, steps, random_start=False):
    x_orig = x.detach()
    eps_t = eps_to_tensor(eps)
    alpha_t = eps_to_tensor(alpha)

    if random_start:
        noise = torch.empty_like(x_orig).uniform_(-1.0, 1.0)
        x_adv = x_orig + noise * eps_t
        x_adv = clamp_normalized(x_adv)
    else:
        x_adv = x_orig.clone().detach()

    for _ in range(int(steps)):
        x_adv.requires_grad_(True)
        logits = model(x_adv)
        loss = criterion(logits, y)

        model.zero_grad(set_to_none=True)
        loss.backward()

        with torch.no_grad():
            x_adv = x_adv + alpha_t * x_adv.grad.sign()
            x_adv = torch.max(torch.min(x_adv, x_orig + eps_t), x_orig - eps_t)
            x_adv = clamp_normalized(x_adv)

    return x_adv.detach()

def bim_attack(model, x, y, eps, alpha, steps):
    return pgd_attack(
        model=model,
        x=x,
        y=y,
        eps=eps,
        alpha=alpha,
        steps=steps,
        random_start=False,
    )

def apply_attack(model, x, y, method, eps, alpha=None, steps=None):
    method = method.lower()
    if method == "fgsm":
        return fgsm_attack(model, x, y, eps)
    if method == "bim":
        if alpha is None or steps is None:
            raise ValueError("Для BIM нужно указать alpha и steps")
        return bim_attack(model, x, y, eps, alpha, steps)
    if method == "pgd":
        if alpha is None or steps is None:
            raise ValueError("Для PGD нужно указать alpha и steps")
        return pgd_attack(model, x, y, eps, alpha, steps, random_start=cfg.PGD_RANDOM_START)
    raise ValueError(f"Неизвестный метод атаки: {method}")

def defense_specs_from_cfg(cfg):
    specs = [{
        "name": "none",
        "label": "Без защиты",
        "blur_kernel": None,
        "quant_levels": None,
    }]
    for k in cfg.BLUR_KERNELS:
        specs.append({
            "name": f"blur_{k}",
            "label": f"Gaussian blur {k}x{k}",
            "blur_kernel": int(k),
            "quant_levels": None,
        })
    for levels in cfg.QUANT_LEVELS:
        specs.append({
            "name": f"quant_{levels}",
            "label": f"Quantization {levels}",
            "blur_kernel": None,
            "quant_levels": int(levels),
        })
    if cfg.INCLUDE_COMBINED_DEFENSES:
        for k in cfg.BLUR_KERNELS:
            for levels in cfg.QUANT_LEVELS:
                specs.append({
                    "name": f"blur_{k}_quant_{levels}",
                    "label": f"Blur {k}x{k} + Quant {levels}",
                    "blur_kernel": int(k),
                    "quant_levels": int(levels),
                })
    return specs

DEFENSE_SPECS = defense_specs_from_cfg(cfg)
pd.DataFrame(DEFENSE_SPECS)

In [8]:
@torch.no_grad()
def predict_batch(model, x):
    logits = model(x)
    probs = torch.softmax(logits, dim=1)
    preds = torch.argmax(probs, dim=1)
    return preds, probs

def gaussian_blur_defense(x, kernel_size):
    x01 = denormalize_batch(x)
    x01_blur = TF.gaussian_blur(x01, kernel_size=[kernel_size, kernel_size])
    return normalize_batch(x01_blur)

def quantize_defense(x, levels):
    if levels < 2:
        raise ValueError("levels должно быть >= 2")
    x01 = denormalize_batch(x)
    x01_q = torch.round(x01 * (levels - 1)) / (levels - 1)
    x01_q = torch.clamp(x01_q, 0.0, 1.0)
    return normalize_batch(x01_q)

def apply_defense(x, blur_kernel=None, quant_levels=None):
    if blur_kernel is None and quant_levels is None:
        return x
    x_out = x
    if blur_kernel is not None:
        x_out = gaussian_blur_defense(x_out, int(blur_kernel))
    if quant_levels is not None:
        x_out = quantize_defense(x_out, int(quant_levels))
    return x_out

def to_eps_label(eps):
    return f"{int(round(float(eps) * 255))}/255"

## 4. Оценка защиты

In [9]:
def evaluate_defenses_for_attack(df, attack_method, eps, alpha=None, steps=None):
    work_df = df.copy().reset_index(drop=True)
    if cfg.MAX_SAMPLES_PER_ATTACK is not None:
        work_df = work_df.sample(min(cfg.MAX_SAMPLES_PER_ATTACK, len(work_df)), random_state=cfg.SEED).reset_index(drop=True)

    loader = make_loader(work_df, shuffle=False)
    detail_rows = []
    timing_rows = []

    for x, y, paths, labels in loader:
        x = x.to(device)
        y = y.to(device)
        batch_size = x.shape[0]

        # clean baseline
        sync_device()
        t0 = time.perf_counter()
        preds_clean, probs_clean = predict_batch(model, x)
        sync_device()
        clean_infer_sec = time.perf_counter() - t0

        # adversarial batch
        sync_device()
        t0 = time.perf_counter()
        x_adv = apply_attack(
            model=model,
            x=x,
            y=y,
            method=attack_method,
            eps=eps,
            alpha=alpha,
            steps=steps,
        )
        sync_device()
        attack_gen_sec = time.perf_counter() - t0

        sync_device()
        t0 = time.perf_counter()
        preds_adv, probs_adv = predict_batch(model, x_adv)
        sync_device()
        adv_infer_sec = time.perf_counter() - t0

        y_np = y.detach().cpu().numpy()
        preds_clean_np = preds_clean.detach().cpu().numpy()
        preds_adv_np = preds_adv.detach().cpu().numpy()
        probs_clean_np = probs_clean.detach().cpu().numpy()
        probs_adv_np = probs_adv.detach().cpu().numpy()

        # для каждой защиты отдельно считаем влияние на clean и adv
        for spec in DEFENSE_SPECS:
            sync_device()
            t0 = time.perf_counter()
            x_clean_def = apply_defense(x, spec["blur_kernel"], spec["quant_levels"])
            x_adv_def = apply_defense(x_adv, spec["blur_kernel"], spec["quant_levels"])
            sync_device()
            defense_sec = time.perf_counter() - t0

            sync_device()
            t0 = time.perf_counter()
            preds_clean_def, probs_clean_def = predict_batch(model, x_clean_def)
            preds_adv_def, probs_adv_def = predict_batch(model, x_adv_def)
            sync_device()
            defense_infer_sec = time.perf_counter() - t0

            preds_clean_def_np = preds_clean_def.detach().cpu().numpy()
            preds_adv_def_np = preds_adv_def.detach().cpu().numpy()
            probs_clean_def_np = probs_clean_def.detach().cpu().numpy()
            probs_adv_def_np = probs_adv_def.detach().cpu().numpy()

            timing_rows.append({
                "attack_method": attack_method.upper(),
                "epsilon": float(eps),
                "epsilon_label": to_eps_label(eps),
                "defense_name": spec["name"],
                "defense_label": spec["label"],
                "n_images": batch_size,
                "attack_gen_sec": attack_gen_sec,
                "clean_infer_sec": clean_infer_sec,
                "adv_infer_sec": adv_infer_sec,
                "defense_sec": defense_sec,
                "defense_infer_sec": defense_infer_sec,
            })

            for i in range(batch_size):
                target_idx = int(y_np[i])
                clean_pred = int(preds_clean_np[i])
                adv_pred = int(preds_adv_np[i])
                clean_def_pred = int(preds_clean_def_np[i])
                adv_def_pred = int(preds_adv_def_np[i])

                detail_rows.append({
                    "attack_method": attack_method.upper(),
                    "epsilon": float(eps),
                    "epsilon_label": to_eps_label(eps),
                    "defense_name": spec["name"],
                    "defense_label": spec["label"],
                    "blur_kernel": spec["blur_kernel"],
                    "quant_levels": spec["quant_levels"],
                    "crop_path": paths[i],
                    "label": labels[i],
                    "target": target_idx,

                    "pred_clean": clean_pred,
                    "pred_clean_label": IDX_TO_CLASS[clean_pred],
                    "pred_attack": adv_pred,
                    "pred_attack_label": IDX_TO_CLASS[adv_pred],
                    "pred_clean_def": clean_def_pred,
                    "pred_clean_def_label": IDX_TO_CLASS[clean_def_pred],
                    "pred_adv_def": adv_def_pred,
                    "pred_adv_def_label": IDX_TO_CLASS[adv_def_pred],

                    "clean_correct_no_defense": int(clean_pred == target_idx),
                    "attack_correct_no_defense": int(adv_pred == target_idx),
                    "clean_correct_with_defense": int(clean_def_pred == target_idx),
                    "adv_correct_with_defense": int(adv_def_pred == target_idx),

                    "attack_success_before_defense": int(adv_pred != target_idx),
                    "recovered_after_defense": int((adv_pred != target_idx) and (adv_def_pred == target_idx)),

                    "p_clean_red": float(probs_clean_np[i][CLASS_TO_IDX["red"]]) if "red" in CLASS_TO_IDX else np.nan,
                    "p_clean_green": float(probs_clean_np[i][CLASS_TO_IDX["green"]]) if "green" in CLASS_TO_IDX else np.nan,
                    "p_attack_red": float(probs_adv_np[i][CLASS_TO_IDX["red"]]) if "red" in CLASS_TO_IDX else np.nan,
                    "p_attack_green": float(probs_adv_np[i][CLASS_TO_IDX["green"]]) if "green" in CLASS_TO_IDX else np.nan,
                    "p_clean_def_red": float(probs_clean_def_np[i][CLASS_TO_IDX["red"]]) if "red" in CLASS_TO_IDX else np.nan,
                    "p_clean_def_green": float(probs_clean_def_np[i][CLASS_TO_IDX["green"]]) if "green" in CLASS_TO_IDX else np.nan,
                    "p_adv_def_red": float(probs_adv_def_np[i][CLASS_TO_IDX["red"]]) if "red" in CLASS_TO_IDX else np.nan,
                    "p_adv_def_green": float(probs_adv_def_np[i][CLASS_TO_IDX["green"]]) if "green" in CLASS_TO_IDX else np.nan,
                })

    detail_df = pd.DataFrame(detail_rows)
    timing_df = pd.DataFrame(timing_rows)
    return detail_df, timing_df

In [ ]:
all_detail_parts = []
all_timing_parts = []

for (method, eps_px), attack_df in sorted(selected_tables.items()):
    eps = eps_px / 255.0

    # alpha/steps можно попытаться прочитать из CSV, а если нет — взять из cfg
    alpha = None
    steps = None
    if "alpha" in attack_df.columns and attack_df["alpha"].notna().any():
        alpha = float(attack_df["alpha"].dropna().iloc[0])
    if "steps" in attack_df.columns and attack_df["steps"].notna().any():
        steps = int(float(attack_df["steps"].dropna().iloc[0]))

    if method == "bim":
        alpha = cfg.BIM_ALPHA if alpha is None else alpha
        steps = cfg.BIM_STEPS if steps is None else steps
    elif method == "pgd":
        alpha = cfg.PGD_ALPHA if alpha is None else alpha
        steps = cfg.PGD_STEPS if steps is None else steps
    else:
        alpha = None
        steps = None

    print(f"Обработка {method.upper()} eps={eps_px}/255 | n={len(attack_df)} | alpha={alpha} | steps={steps}")
    detail_df_part, timing_df_part = evaluate_defenses_for_attack(
        df=attack_df,
        attack_method=method,
        eps=eps,
        alpha=alpha,
        steps=steps,
    )
    all_detail_parts.append(detail_df_part)
    all_timing_parts.append(timing_df_part)

detail_df = pd.concat(all_detail_parts, ignore_index=True)
timing_df = pd.concat(all_timing_parts, ignore_index=True)

detail_path = OUT_DIR / "defense_detail_predictions.csv"
timing_path = OUT_DIR / "defense_timing_detail.csv"

detail_df.to_csv(detail_path, index=False, encoding="utf-8-sig")
timing_df.to_csv(timing_path, index=False, encoding="utf-8-sig")

print("detail_df:", detail_df.shape)
print("timing_df:", timing_df.shape)
print("Сохранено:", detail_path)
print("Сохранено:", timing_path)
display(detail_df.head())

In [ ]:
base_df = (
    detail_df[detail_df["defense_name"] == "none"]
    .groupby(["attack_method", "epsilon", "epsilon_label"], as_index=False)
    .agg(
        n_samples=("crop_path", "count"),
        clean_accuracy_no_defense=("clean_correct_no_defense", "mean"),
        attacked_accuracy_no_defense=("attack_correct_no_defense", "mean"),
        attack_success_rate=("attack_success_before_defense", "mean"),
        broken_before_defense=("attack_success_before_defense", "sum"),
    )
)

summary_df = (
    detail_df
    .groupby(
        ["attack_method", "epsilon", "epsilon_label", "defense_name", "defense_label", "blur_kernel", "quant_levels"],
        as_index=False,
        dropna=False
    )
    .agg(
        n_samples=("crop_path", "count"),
        clean_accuracy_with_defense=("clean_correct_with_defense", "mean"),
        defended_accuracy_after_attack=("adv_correct_with_defense", "mean"),
        recovered_after_defense=("recovered_after_defense", "sum"),
    )
)

base_df_renamed = base_df.rename(columns={"n_samples": "n_samples_base"})

summary_df = summary_df.merge(
    base_df_renamed,
    on=["attack_method", "epsilon", "epsilon_label"],
    how="left"
)

summary_df["recovery_gain_abs"] = summary_df["defended_accuracy_after_attack"] - summary_df["attacked_accuracy_no_defense"]
summary_df["recovery_gain_pct_points"] = summary_df["recovery_gain_abs"] * 100.0
summary_df["clean_drop_abs"] = summary_df["clean_accuracy_no_defense"] - summary_df["clean_accuracy_with_defense"]
summary_df["clean_drop_pct_points"] = summary_df["clean_drop_abs"] * 100.0

summary_df["recovered_share_among_broken"] = np.where(
    summary_df["broken_before_defense"] > 0,
    summary_df["recovered_after_defense"] / summary_df["broken_before_defense"],
    0.0,
)

timing_summary_df = (
    timing_df
    .groupby(["attack_method", "epsilon", "epsilon_label", "defense_name", "defense_label"], as_index=False)
    .agg(
        n_images=("n_images", "sum"),
        attack_gen_sec=("attack_gen_sec", "sum"),
        clean_infer_sec=("clean_infer_sec", "sum"),
        adv_infer_sec=("adv_infer_sec", "sum"),
        defense_sec=("defense_sec", "sum"),
        defense_infer_sec=("defense_infer_sec", "sum"),
    )
)

timing_summary_df["attack_gen_ms_per_roi"] = timing_summary_df["attack_gen_sec"] * 1000 / timing_summary_df["n_images"]
timing_summary_df["defense_ms_per_roi"] = timing_summary_df["defense_sec"] * 1000 / timing_summary_df["n_images"]
timing_summary_df["defense_infer_ms_per_roi"] = timing_summary_df["defense_infer_sec"] * 1000 / timing_summary_df["n_images"]

summary_df = summary_df.merge(
    timing_summary_df[
        ["attack_method", "epsilon", "epsilon_label", "defense_name", "defense_label",
         "attack_gen_ms_per_roi", "defense_ms_per_roi", "defense_infer_ms_per_roi"]
    ],
    on=["attack_method", "epsilon", "epsilon_label", "defense_name", "defense_label"],
    how="left"
)

summary_path = OUT_DIR / "defense_summary.csv"
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

display(summary_df.sort_values(["attack_method", "epsilon", "defense_name"]).head(20))
print("Сохранено:", summary_path)

## 5. Компактные таблицы для текста работы

In [ ]:
report_df = summary_df.copy()

report_df["clean_accuracy_no_defense"] = report_df["clean_accuracy_no_defense"].round(4)
report_df["clean_accuracy_with_defense"] = report_df["clean_accuracy_with_defense"].round(4)
report_df["attacked_accuracy_no_defense"] = report_df["attacked_accuracy_no_defense"].round(4)
report_df["defended_accuracy_after_attack"] = report_df["defended_accuracy_after_attack"].round(4)
report_df["recovery_gain_abs"] = report_df["recovery_gain_abs"].round(4)
report_df["recovery_gain_pct_points"] = report_df["recovery_gain_pct_points"].round(2)
report_df["clean_drop_abs"] = report_df["clean_drop_abs"].round(4)
report_df["clean_drop_pct_points"] = report_df["clean_drop_pct_points"].round(2)
report_df["recovered_share_among_broken"] = report_df["recovered_share_among_broken"].round(4)
report_df["defense_ms_per_roi"] = report_df["defense_ms_per_roi"].round(3)
report_df["defense_infer_ms_per_roi"] = report_df["defense_infer_ms_per_roi"].round(3)

report_cols = [
    "attack_method", "epsilon_label", "defense_label", "n_samples",
    "clean_accuracy_no_defense", "clean_accuracy_with_defense",
    "attacked_accuracy_no_defense", "defended_accuracy_after_attack",
    "recovery_gain_pct_points", "clean_drop_pct_points",
    "recovered_share_among_broken",
    "defense_ms_per_roi", "defense_infer_ms_per_roi",
]

report_df = report_df[report_cols].rename(columns={
    "attack_method": "Атака",
    "epsilon_label": "epsilon",
    "defense_label": "Защита",
    "n_samples": "Число ROI",
    "clean_accuracy_no_defense": "Accuracy clean без защиты",
    "clean_accuracy_with_defense": "Accuracy clean с защитой",
    "attacked_accuracy_no_defense": "Accuracy attack без защиты",
    "defended_accuracy_after_attack": "Accuracy attack+defense",
    "recovery_gain_pct_points": "Восстановление, п.п.",
    "clean_drop_pct_points": "Потеря на clean, п.п.",
    "recovered_share_among_broken": "Доля восстановленных среди сломанных",
    "defense_ms_per_roi": "Время защиты, мс/ROI",
    "defense_infer_ms_per_roi": "Инференс после защиты, мс/ROI",
})

report_path = OUT_DIR / "defense_report_table.csv"
report_df.to_csv(report_path, index=False, encoding="utf-8-sig")

display(report_df.sort_values(["Атака", "epsilon", "Защита"]))
print("Сохранено:", report_path)